In [1]:
from pathlib import Path
import logging
import math
import shutil
import subprocess
import textwrap
import traceback
import zipfile
import nemosis
import pandas as pd
import io
import openpyxl
import pyarrow as pa


logging.getLogger("nemosis").setLevel(logging.WARNING)

In [ ]:
"""
A generic scheduler function to fetch datasource data by months
and skip re pulling existing months
"""

def _month_compiler(
        datasource_fetch_function: callable,
        start_date:pd.Timestamp,
        end_date:pd.Timestamp,
        datasource_file_path:Path,
        fallback_fetch_function: callable = None
        ):
    
    datasource_file_year_month_range = set()
    # Check if the datasource file already exists, if so, check which years/months it has
    if datasource_file_path.exists() and datasource_file_path.stat().st_size > 0:
        # Extract only the SETTLEMENTDATE index to check which months are already present
        datasource_file_date_range = pd.read_parquet(datasource_file_path, columns=[]).index
        # Create a 'set' of unique year-months from datasource_file_date_range
        datasource_file_year_month_range = set(datasource_file_date_range.to_period("M"))
        print(f"  Found {len(datasource_file_year_month_range)} already-processed month(s) — will skip.", flush=True)

    # Set a var for total required data range
    datasource_required_total_range = pd.date_range(start_date, end_date - pd.Timedelta(days=1), freq="MS")

    # Loop through the datasource_required_total_range
    for i, month in enumerate(datasource_required_total_range):
        # Month in datasource_file_year_month_range so skip iteration
        if month.to_period("M") in datasource_file_year_month_range:
            print(f"{i + 1:3d}/{len(datasource_required_total_range)} {month:%Y-%m} skipping (already processed).", flush=True)
            continue
        # Month NOT in datasource_file_year_month_range so call datasource fetch function
        else:
            # Define the start of the calander month  
            current_month_start = month.strftime("%Y/%m/%d %H:%M:%S")
            # Define the end of the calander month
            current_month_end = (month + pd.offsets.MonthEnd(0) + pd.Timedelta(days=1)).strftime("%Y/%m/%d %H:%M:%S")
            
            print(f"{i + 1:3d}/{len(datasource_required_total_range)} {month:%Y-%m} fetching...", flush=True)
            
            # Call the datasource fetch function for the current month
            try:
                datasource_data = datasource_fetch_function(current_month_start, current_month_end)
            except RuntimeError as e:
                if "404" in str(e) or "not available from MMS Historical Data SQL Loader" in str(e):
                    if fallback_fetch_function is not None:
                        print(f"  {month:%Y-%m} primary 404 — falling back to per-run archive...", flush=True)
                        try:
                            datasource_data = fallback_fetch_function(current_month_start, current_month_end)
                            print(f"  {month:%Y-%m} backup succeeded.", flush=True)
                        except Exception as backup_err:
                            print(f"  {month:%Y-%m} SKIPPED — backup also failed: {backup_err}", flush=True)
                            continue
                    else:
                        print(f"  {month:%Y-%m} SKIPPED — 404 Not Found (file unavailable on AEMO server).", flush=True)
                        continue
                else:
                    raise

            # Merge with existing datasource file if it exists
            if datasource_file_path.exists():
                datasource_file = pd.read_parquet(datasource_file_path)
                datasource_data = pd.concat([datasource_file, datasource_data])
                datasource_data = datasource_data[~datasource_data.index.duplicated(keep="last")].sort_index()

            # Ensure the index name is set correctly
            datasource_data.index.name = "Date"
            # Save after every new month
            datasource_data.to_parquet(datasource_file_path)

            # Empty the temporary cache (Deletes all files and subdirectories)
            for entry in Path("Pre_processing/temporary_cache").iterdir():
                if entry.is_dir():
                    shutil.rmtree(entry)
                else:
                    entry.unlink()

            print(f"  {month:%Y-%m} saved & raw files deleted.", flush=True)

    return pd.read_parquet(datasource_file_path)


In [3]:
"""
A generic function to fetch datasource data in one shot
"""

def _one_shot_compiler(
        datasource_fetch_function: callable,
        start_date:pd.Timestamp,
        end_date:pd.Timestamp,
        datasource_file_path:Path
        ):
    
    
        datasource_data = datasource_fetch_function(start_date, end_date)
        datasource_data.index.name = "SETTLEMENTDATE"
        datasource_data.to_parquet(datasource_file_path)

        return pd.read_parquet(datasource_file_path)


In [4]:
"""
Datasource 1
"""
def _dispatch_price(start: str, end: str):
    region_columns = {"NSW1": "nsw_price", "QLD1": "qld_price", "VIC1": "vic_price", "SA1": "sa_price"}

    def _API_call():

        API_response = nemosis.dynamic_data_compiler(
            start_time=start,
            end_time=end,
            table_name="DISPATCHPRICE",
            raw_data_location="Pre_processing/temporary_cache",
            select_columns=["SETTLEMENTDATE", "REGIONID", "RRP"],
            filter_cols=["REGIONID"],
            filter_values=[list(region_columns.keys())],
            fformat="feather",
            keep_csv=False,
        )
        
        return API_response

    def _clean_up(API_response):
        # Handle conversion of data types
        API_response["SETTLEMENTDATE"] = pd.to_datetime(API_response["SETTLEMENTDATE"])
        API_response["RRP"] = pd.to_numeric(API_response["RRP"], errors="coerce")

        # Handle data granularity
        API_response = (
            API_response.groupby(["SETTLEMENTDATE", "REGIONID"])["RRP"]
            .mean()
            .unstack("REGIONID")
            .resample("5min").mean()
            .asfreq("5min")
        )

        # Rename columns
        API_response = API_response.rename(columns=region_columns)

        return API_response

    data = _API_call()
    data_clean = _clean_up(data)
    data_clean.index.name = "Date"
    return data_clean


In [5]:
"""
Datasource 2
"""
def _dispatch_region_sum(start: str, end: str):
    RAW_FIELDS    = ["TOTALDEMAND", "AVAILABLEGENERATION", "NETINTERCHANGE", "DEMANDFORECAST", "DISPATCHABLEGENERATION"]
    REGION_SUFFIX = {"NSW1": "_nsw", "QLD1": "_qld", "VIC1": "_vic", "SA1": "_sa"}

    def _API_call():
        
        API_response = nemosis.dynamic_data_compiler(
            start_time=start,
            end_time=end,
            table_name="DISPATCHREGIONSUM",
            raw_data_location="Pre_processing/temporary_cache",
            select_columns=["SETTLEMENTDATE", "REGIONID"] + RAW_FIELDS,
            filter_cols=["REGIONID"],
            filter_values=[list(REGION_SUFFIX.keys())],
            fformat="feather",
            keep_csv=False,
        )
        
        return API_response

    def _clean_up(API_response):

        # Handle conversion of data types
        OUTPUT_FIELDS = ["demand", "avail_gen", "interchange", "demand_forecast", "dispatch_gen"]

        API_response["SETTLEMENTDATE"] = pd.to_datetime(API_response["SETTLEMENTDATE"])

        for col in RAW_FIELDS:
            API_response[col] = pd.to_numeric(API_response[col], errors="coerce")

        # Handle data granularity / transform from long format
        region_frames = []
        for region, suffix in REGION_SUFFIX.items():
            rdf = API_response[API_response["REGIONID"] == region][["SETTLEMENTDATE"] + RAW_FIELDS].copy()
            rdf = (
                rdf.set_index("SETTLEMENTDATE")
                .sort_index()
                .resample("5min")
                .mean(numeric_only=True)
            )
            rdf.columns = [f"{name}{suffix}" for name in OUTPUT_FIELDS]
            region_frames.append(rdf)
            
        API_response = pd.concat(region_frames, axis=1).sort_index()

        return API_response

    data = _API_call()
    data_clean = _clean_up(data)
    data_clean.index.name = "Date"
    return data_clean




In [6]:
def _nem_registration_and_exemption_list():

    def _API_call():
        cache_path = Path("Pre_processing/temporary_cache") / "NEM Registration and Exemption List.xlsx"
        url = "https://www.aemo.com.au/-/media/files/electricity/nem/participant_information/nem-registration-and-exemption-list.xlsx"
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as resp, open(cache_path, "wb") as f:
            f.write(resp.read())
        try:
            df = pd.read_excel(cache_path, sheet_name="PU and Scheduled Loads", engine="openpyxl", header=0)
        finally:
            cache_path.unlink(missing_ok=True)
        return df
            
    def _clean_up(df):
        df = df[df["Dispatch Type"].astype(str).str.upper() != "LOAD"]
        df = df[df["DUID"].astype(str).str.upper() != "-"].copy()
        df["DUID"] = df["DUID"].astype(str).str.strip()
        df["Fuel Source - Primary"] = df["Fuel Source - Primary"].fillna("").astype(str).str.strip()
        df = df[df["Fuel Source - Primary"] != ""].copy()
        df = df[df["Category"].astype(str).str.strip().str.upper() == "MARKET"].copy()
        return df

    def _assign_fuel_type(df):
        mask = df["Fuel Source - Primary"].str.lower().str.contains("biomass", na=False)
        df.loc[mask, "Fuel Source - Primary"] = "Biomass"

        fossil_mask = df["Fuel Source - Primary"].astype(str).str.strip().str.lower().eq("fossil")
        coal_mask = df["Fuel Source - Descriptor"].astype(str).str.strip().isin({"Black Coal", "Brown Coal", "Coal Seam Methane"})

        df.loc[fossil_mask & coal_mask, "Fuel Source - Primary"] = "Coal"
        df.loc[fossil_mask & ~coal_mask, "Fuel Source - Primary"] = "Gas"

        df["Region"] = (df["Region"].astype(str).str.strip().str.replace(r"1$", "", regex=True))

        return df[["Region", "Participant", "Station Name", "DUID","Classification","Fuel Source - Primary"]].reset_index(drop=True)
       
    df = _API_call()
    df = _clean_up(df)
    df = _assign_fuel_type(df)
    df.to_parquet("Processed_data/0_nem_duid_mapping.parquet")
    return df

if False:
    _nem_registration_and_exemption_list()


In [7]:
"""
Datasource 3
"""
def _generation_fuel(start: str, end: str):

    def _API_call(duid_to_col):
        API_response = nemosis.dynamic_data_compiler(
            start_time=start,
            end_time=end,
            table_name="DISPATCH_UNIT_SCADA",
            raw_data_location="Pre_processing/temporary_cache",
            select_columns=["SETTLEMENTDATE", "DUID", "SCADAVALUE"],
            filter_cols=["DUID"],
            filter_values=[sorted(duid_to_col.keys())],
            fformat="feather",
            keep_csv=False,
        )
        return API_response

    def _clean_up(API_response, duid_to_col):
        battery_duids = {duid for duid, col in duid_to_col.items() if col.startswith("battery_mw_")}

        API_response["SETTLEMENTDATE"] = pd.to_datetime(API_response["SETTLEMENTDATE"])
        API_response["SCADAVALUE"]     = pd.to_numeric(API_response["SCADAVALUE"], errors="coerce")
        API_response["_col"]           = API_response["DUID"].map(duid_to_col)
        API_response = API_response[API_response["_col"].notna()].copy()

        API_response["_target"] = API_response["_col"]
        API_response["_value"]  = API_response["SCADAVALUE"].clip(lower=0)
        is_battery = API_response["DUID"].isin(battery_duids)
        if is_battery.any():
            charge    = is_battery & (API_response["SCADAVALUE"] < 0)
            discharge = is_battery & (API_response["SCADAVALUE"] >= 0)
            API_response.loc[charge,    "_target"] = API_response.loc[charge,    "_col"].str.replace("battery_mw_", "battery_charge_mw_",    regex=False)
            API_response.loc[charge,    "_value"]  = -API_response.loc[charge,   "SCADAVALUE"]
            API_response.loc[discharge, "_target"] = API_response.loc[discharge, "_col"].str.replace("battery_mw_", "battery_discharge_mw_", regex=False)
            API_response.loc[discharge, "_value"]  = API_response.loc[discharge, "SCADAVALUE"].clip(lower=0)

        API_response = (
            API_response.groupby(["SETTLEMENTDATE", "_target"])["_value"]
            .sum()
            .unstack("_target")
            .resample("5min").mean()
            .fillna(0)
        )

        # Ensure every fuel×region combination is present, even if always 0.
        all_regions  = sorted({col.split("_mw_")[1] for col in duid_to_col.values()})
        all_fuels    = sorted({col.split("_mw_")[0] for col in duid_to_col.values() if not col.startswith("battery_mw_")})
        has_battery  = any(col.startswith("battery_mw_") for col in duid_to_col.values())
        non_battery  = [f"{fuel}_mw_{region}" for fuel in all_fuels for region in all_regions]
        battery_cols = [f"battery_{d}_mw_{region}" for d in ("charge", "discharge") for region in all_regions] if has_battery else []
        all_expected = sorted(set(non_battery + battery_cols), key=lambda c: (c.split("_mw_")[1], c.split("_mw_")[0]))
        API_response = API_response.reindex(columns=all_expected, fill_value=0.0)

        return API_response

    duid_map = pd.read_parquet("Processed_data/0_nem_duid_mapping.parquet")
    duid_map = duid_map[~duid_map["Region"].str.upper().isin({"TAS"})]
    def _fuel_key(fuel: str) -> str:
        f = fuel.lower().strip()
        return "battery" if "battery" in f else f
    duid_to_col = {
        row["DUID"]: f"{_fuel_key(row['Fuel Source - Primary'])}_mw_{row['Region'].lower()}"
        for _, row in duid_map.iterrows()
    }
    data = _API_call(duid_to_col)
    data_clean = _clean_up(data, duid_to_col)
    data_clean.index.name = "Date"
    return data_clean


In [8]:
"""
Datasource 4
"""

def _STTM_DWGM(start: str, end: str):
    

    def _load_1():
        request = urllib.request.Request(
            "https://www.aemo.com.au/-/media/files/gas/sttm/data/sttm-price-and-withdrawals.xlsx",
            headers={"User-Agent": "Mozilla/5.0"},
        )

        with urllib.request.urlopen(request, timeout=60) as response:
            data = openpyxl.load_workbook(io.BytesIO(response.read()), read_only=True, data_only=True)

        return data
    

    def _load_2():
        request = urllib.request.Request(
            "https://www.aemo.com.au/-/media/files/gas/dwgm/dwgm-prices-and-demand.xlsx",
            headers={"User-Agent": "Mozilla/5.0"},
        )

        with urllib.request.urlopen(request, timeout=60) as response:
            data = openpyxl.load_workbook(io.BytesIO(response.read()), read_only=True, data_only=True)

        return data
    

    def _process(data1, data2):

        idx = pd.date_range(start=start, end=end, freq="5min", name="Date")

        sheets = {
            "SYD price and withdrawals": "gas_price_nsw",
            "ADL price and withdrawals": "gas_price_sa",
            "BRI price and withdrawals": "gas_price_qld",
        }

        data_clean = {}
        for sheet_name, col_name in sheets.items():
            rows = [(pd.Timestamp(r[0]), float(r[1])) for r in data1[sheet_name].iter_rows(min_row=2, values_only=True) if r[0] is not None and r[1] is not None]
            daily = pd.Series(dict(rows))
            data_clean[col_name] = daily.reindex(idx.normalize()).set_axis(idx).ffill().bfill()

        # DWGM (VIC) — 5 sub-daily intervals, ffill within each ~4-hour horizon
        rows_vic = [(pd.Timestamp(r[0]) + pd.Timedelta(hours=int(r[1])), float(r[2])) for r in data2["Prices"].iter_rows(min_row=2, values_only=True) if r[0] is not None and r[1] is not None and r[2] is not None]
        dwgm = pd.Series(dict(rows_vic), name="gas_price_vic").sort_index()
        data_clean["gas_price_vic"] = dwgm.reindex(idx, method="ffill").bfill()

        return pd.DataFrame(data_clean, index=idx)
    


    data1, data2 = _load_1(), _load_2()
    data_clean = _process(data1,data2)
    return data_clean


In [9]:
"""
Datasource 5
"""

def _weather(start: str, end: str):
    
    def _load():
        sydney    = pd.read_parquet("Pre_processing/weather_data/NSW weather.parquet")
        brisbane  = pd.read_parquet("Pre_processing/weather_data/QLD weather.parquet")
        melbourne = pd.read_parquet("Pre_processing/weather_data/VIC weather.parquet")
        adelaide  = pd.read_parquet("Pre_processing/weather_data/SA weather.parquet")

        return sydney, brisbane, melbourne, adelaide

    def _process(data: pd.DataFrame, city: str) -> pd.DataFrame:
        data["datetime"] = pd.to_datetime(data["datetime"])
        data = data.set_index("datetime").sort_index()
        data = data[~data.index.duplicated(keep="first")]
        data = data.rename(columns={col: f"{str(col).strip().lower().replace(' ', '')}_{city}" for col in data.columns})
        data = data.apply(pd.to_numeric, errors="coerce")
        data = data.dropna(axis=1, how="all")
        return data.resample(f"{5}min").interpolate(method="time")
    
    sydney, brisbane, melbourne, adelaide = _load()

    sydney    = _process(sydney, "sydney")
    brisbane  = _process(brisbane, "brisbane")
    melbourne = _process(melbourne, "melbourne")
    adelaide  = _process(adelaide,  "adelaide")

   
    data_clean = pd.concat([sydney, brisbane, melbourne, adelaide], axis=1)
    data_clean.index.name = "Date"
   
    return data_clean

   


In [10]:
def _nemseer_pull(
        start_date: pd.Timestamp,
        end_date: pd.Timestamp,
        datasource_file_path: Path,
        nemseer_forecast_type: str,
        nemseer_table_name: str,
        value_cols: list,
        run_col: str,
        interval_col: str,
        entity_col: str,
        ):

    def _repeat_logic(start: str, end: str, _use_backup: bool = False, _use_mmsdm: bool = False):

        modified_start = (pd.Timestamp(start) - pd.Timedelta(days=1)).strftime("%Y/%m/%d %H:%M")

        def _API_call_default():
            # nemseer call requires two separate temp folders
            Path("Pre_processing/temporary_cache/raw").mkdir(parents=True, exist_ok=True)
            Path("Pre_processing/temporary_cache/processed").mkdir(parents=True, exist_ok=True)

            def get_special_dates():
                # Compute max allowed forecasted_end: nemseer caps at next-trading-day 04:00, Trading days run 04:00–04:00 AEST
                run_end_ts = pd.Timestamp(end[:16])
                current_trading_day_end = run_end_ts.normalize() + pd.Timedelta(hours=4)
                if run_end_ts >= current_trading_day_end:
                    current_trading_day_end += pd.Timedelta(days=1)
                return current_trading_day_end.strftime("%Y/%m/%d %H:%M")

            forecasted_end = get_special_dates()

            subprocess_code = textwrap.dedent(f"""
                import logging
                logging.getLogger("nemseer").setLevel(logging.WARNING)
                import pandas as pd
                pd.options.future.infer_string = False
                import nemseer, sys
                data = nemseer.compile_data(
                    run_start={modified_start!r},
                    run_end={end[:16]!r},
                    forecasted_start={modified_start!r},
                    forecasted_end={forecasted_end!r},
                    raw_cache="Pre_processing/temporary_cache/raw",
                    processed_cache="Pre_processing/temporary_cache/processed",
                    forecast_type={nemseer_forecast_type!r},
                    tables=[{nemseer_table_name!r}],
                )
                df = data[{nemseer_table_name!r}]
                if df is None or df.empty:
                    raise ValueError("nemseer returned no data")
                df.reset_index().to_csv(sys.stdout, index=False)
            """)
            
            path = "/home/ec2-user/venv311/bin/python"  # EC2 Linux: python3.11 venv

            result = subprocess.run(
                [path, "-c", subprocess_code],
                capture_output=True, text=True, cwd=Path.cwd(),
            )

            # Needs this to actually display errors from the subprocess
            if result.returncode != 0 or not result.stdout.strip():
                raise RuntimeError(f"nemseer subprocess failed:\n{result.stderr}")

            API_response = pd.read_csv(io.StringIO(result.stdout))
            return API_response

        def _API_call_backup():
            """
            Fallback: fetches pre-dispatch data from the NEMWeb weekly archive at
            nemweb.com.au/Reports/Archive/PreDispatch_Reports/.
            Files are named PUBLIC_PREDISPATCH_YYYYMMDD_YYYYMMDD.zip (one per week).
            Scrapes the directory listing, downloads weekly ZIPs that overlap
            [modified_start, end), parses AEMO's I/D CSV record format, and returns
            a DataFrame in the same shape as _API_call_default.
            Only PREDISPATCH forecast type is supported.
            Note: the archive retains roughly the last 12 months of data.

            Archive format notes (differs from nemseer):
              - Table identifier is in parts[1] (DataSource), not parts[2] (TableName)
              - Column names differ: PREDISPATCHSEQNO → run_col, PERIODID → interval_col
              - Values are quoted CSV, so csv.reader is used instead of str.split
            """
            import re
            import csv as csv_mod

            ARCHIVE_ROOTS = {
                "PREDISPATCH": "https://nemweb.com.au/Reports/Archive/PreDispatch_Reports/",
            }
            if nemseer_forecast_type not in ARCHIVE_ROOTS:
                raise RuntimeError(
                    f"Per-run backup not supported for forecast type '{nemseer_forecast_type}'. "
                    "Only PREDISPATCH is implemented."
                )

            root         = ARCHIVE_ROOTS[nemseer_forecast_type]
            window_start = pd.Timestamp(modified_start[:16])
            window_end   = pd.Timestamp(end[:16])

            # Scrape directory listing to find weekly ZIPs that overlap the window
            req = urllib.request.Request(root, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=60) as resp:
                html = resp.read().decode("utf-8", errors="replace")

            zip_urls = []
            for href in re.findall(r'href=["\']([^"\']*\.zip)["\']', html, re.IGNORECASE):
                fname = href.rsplit("/", 1)[-1]
                # Filename: PUBLIC_PREDISPATCH_YYYYMMDD_YYYYMMDD.zip
                m = re.search(r"PUBLIC_PREDISPATCH_(\d{8})_(\d{8})\.zip", fname, re.IGNORECASE)
                if not m:
                    continue
                try:
                    zip_start = pd.to_datetime(m.group(1), format="%Y%m%d")
                    zip_end   = pd.to_datetime(m.group(2), format="%Y%m%d") + pd.Timedelta(days=1)
                except Exception:
                    continue
                # Include the ZIP if its date range overlaps with [window_start, window_end)
                if zip_start < window_end and zip_end > window_start:
                    full_url = href if href.startswith("http") else root + fname
                    zip_urls.append((zip_start, full_url))

            if not zip_urls:
                raise RuntimeError(
                    f"Backup: no weekly ZIP files found overlapping {window_start} to "
                    f"{window_end} at {root}. The archive only retains ~12 months of data."
                )

            print(f"    Backup archive: found {len(zip_urls)} weekly ZIP(s) — downloading and parsing...", flush=True)

            # AEMO archive uses different table identifiers (in parts[1]) and column names
            # than nemseer. Map nemseer table name → (archive_table_id, {archive_col: nemseer_col})
            PREDISPATCH_TABLE_MAP = {
                "PRICE":             ("PDREGION",       {"PREDISPATCHSEQNO": run_col, "PERIODID": interval_col}),
                "REGIONSUM":         ("PDREGION",       {"PREDISPATCHSEQNO": run_col, "PERIODID": interval_col}),
                "INTERCONNECTORRES": ("PDINTERCONNECT", {"PREDISPATCHSEQNO": run_col, "PERIODID": interval_col}),
            }
            table_entry      = PREDISPATCH_TABLE_MAP.get(nemseer_table_name.upper())
            archive_table_id = table_entry[0] if table_entry else nemseer_table_name.upper()
            archive_col_map  = table_entry[1] if table_entry else {}  # archive_col → nemseer_col

            # Build archive_want: archive_col_upper → nemseer_col_name (used when extracting D rows)
            nemseer_cols       = [run_col, interval_col, entity_col] + value_cols
            nemseer_to_archive = {v: k for k, v in archive_col_map.items()}
            archive_want       = {nemseer_to_archive.get(c, c).upper(): c for c in nemseer_cols}

            all_rows          = []
            seen_tables       = set()
            first_csv_preview = []

            def _parse_text(text):
                header      = None
                col_indices = {}
                for row in csv_mod.reader(text.splitlines()):
                    if not row:
                        continue
                    rec = row[0].upper()
                    if rec == "I":
                        # Track both identifier positions for diagnostics
                        if len(row) > 1: seen_tables.add(row[1].upper())
                        if len(row) > 2: seen_tables.add(row[2].upper())
                    if len(row) < 5:
                        continue
                    # Archive stores table ID in parts[1] (DataSource), not parts[2] (TableName)
                    table_here = (
                        row[1].upper() == archive_table_id or
                        row[2].upper() == archive_table_id
                    )
                    if rec == "I" and table_here:
                        header      = [c.upper() for c in row[4:]]
                        col_indices = {col: i for i, col in enumerate(header)}
                    elif rec == "D" and col_indices and table_here:
                        vals = row[4:]
                        r    = {
                            orig: vals[col_indices[up]]
                            for up, orig in archive_want.items()
                            if up in col_indices and col_indices[up] < len(vals)
                        }
                        if len(r) == len(archive_want):
                            all_rows.append(r)

            def _extract_recursive(zf, depth=0):
                """
                Recursively parse CSVs from a ZIP, handling nested ZIPs up to depth 3.
                AEMO archive structure can be:
                  weekly ZIP → per-run ZIPs → CSVs  (2 levels), or
                  weekly ZIP → daily ZIPs → per-run ZIPs → CSVs  (3 levels).
                """
                for name in zf.namelist():
                    if name.upper().endswith(".CSV"):
                        try:
                            with zf.open(name) as f:
                                text = f.read().decode("utf-8", errors="replace")
                            # Capture the first 8 lines of the very first CSV seen for diagnostics
                            if not first_csv_preview:
                                first_csv_preview.extend(text.splitlines()[:8])
                            _parse_text(text)
                        except Exception as e:
                            print(f"      Warning: could not parse CSV {name}: {e}", flush=True)
                    elif name.upper().endswith(".ZIP") and depth < 3:
                        try:
                            with zf.open(name) as inner_bytes:
                                with zipfile.ZipFile(io.BytesIO(inner_bytes.read())) as inner_zf:
                                    _extract_recursive(inner_zf, depth + 1)
                        except Exception as e:
                            print(f"      Warning: could not open nested ZIP {name}: {e}", flush=True)

            for file_num, (_, url) in enumerate(sorted(zip_urls), 1):
                print(f"    Backup archive: downloading weekly ZIP {file_num}/{len(zip_urls)}...", flush=True)
                try:
                    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
                    with urllib.request.urlopen(req, timeout=120) as resp:
                        zdata = resp.read()
                    with zipfile.ZipFile(io.BytesIO(zdata)) as zf:
                        _extract_recursive(zf)
                except Exception as e:
                    print(f"      Warning: failed to download/process {url.rsplit('/', 1)[-1]}: {e}", flush=True)

            if not all_rows:
                preview_str = "\n".join(first_csv_preview) if first_csv_preview else "(no CSVs reached)"
                raise RuntimeError(
                    f"Backup archive: no rows parsed for table '{nemseer_table_name}' "
                    f"(archive ID: '{archive_table_id}') between {window_start} and {window_end}.\n"
                    f"  Tables found in I-records: {sorted(t for t in seen_tables if t)}\n"
                    f"  Archive want columns: {sorted(archive_want)}\n"
                    f"  First CSV preview:\n{preview_str}"
                )

            return pd.DataFrame(all_rows)

        def _API_call_mmsdm():
            """
            Second fallback: AEMO MMSDM monthly historical archive at
            nemweb.com.au/Data_Archive/Wholesale_Electricity/MMSDM/.
            Covers all historical months. Supports PREDISPATCH (PRICE, REGIONSUM,
            INTERCONNECTORRES) and PDPASA (REGIONSOLUTION).

            PREDISPATCH files: LASTCHANGED floored to 30-min is used as the run
            timestamp (no dedicated run col in MMSDM); DATETIME is the interval col.
            PDPASA files: RUN_DATETIME and INTERVAL_DATETIME are directly available.
            """
            import re as _re, csv as _csv

            # Maps (forecast_type, table_name) → MMSDM file search keyword (matched against HREF)
            MMSDM_FILE_KEY = {
                ("PREDISPATCH", "PRICE"):             "PREDISPATCHPRICE",
                ("PREDISPATCH", "REGIONSUM"):         "PREDISPATCHREGIONSUM",
                ("PREDISPATCH", "INTERCONNECTORRES"): "PREDISPATCHINTERCONNECTORRES",
                ("PDPASA",      "REGIONSOLUTION"):    "PDPASA_REGIONSOLUTION",
            }
            # row[1] value in MMSDM D/I records for each forecast type
            MMSDM_SOURCE_ID = {
                "PREDISPATCH": "PREDISPATCH",
                "PDPASA":      "PDPASA",
            }
            # MMSDM column to use as run timestamp, and whether to floor it to 30-min
            # PREDISPATCH has no dedicated run col; LASTCHANGED is the closest proxy
            MMSDM_RUN_COL = {
                "PREDISPATCH": ("LASTCHANGED",    True),   # (col_name, floor_to_30min)
                "PDPASA":      ("RUN_DATETIME",   False),
            }
            # MMSDM column name for the forecast interval timestamp
            MMSDM_INTERVAL_COL = {
                "PREDISPATCH": "DATETIME",
                "PDPASA":      "INTERVAL_DATETIME",
            }

            key        = (nemseer_forecast_type.upper(), nemseer_table_name.upper())
            search_key = MMSDM_FILE_KEY.get(key)
            if search_key is None:
                raise RuntimeError(
                    f"MMSDM backup not supported for ({nemseer_forecast_type}, {nemseer_table_name}). "
                    f"Supported combinations: {list(MMSDM_FILE_KEY.keys())}"
                )

            mmsdm_source_id          = MMSDM_SOURCE_ID[nemseer_forecast_type.upper()]
            mmsdm_run_col_name, floor_run = MMSDM_RUN_COL[nemseer_forecast_type.upper()]
            mmsdm_interval_col_name  = MMSDM_INTERVAL_COL[nemseer_forecast_type.upper()]

            month_ts = pd.Timestamp(start[:16]).replace(day=1)
            year, month = month_ts.year, month_ts.month

            base_url = (
                f"https://nemweb.com.au/Data_Archive/Wholesale_Electricity/MMSDM/"
                f"{year}/MMSDM_{year}_{month:02d}/MMSDM_Historical_Data_SQLLoader/DATA/"
            )
            req = urllib.request.Request(base_url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=30) as resp:
                html = resp.read().decode("utf-8", errors="replace")

            links      = _re.findall(r'HREF="(/[^"]+)"', html)
            match_link = next(
                (l for l in links if search_key.upper() in l.upper() and l.upper().endswith(".ZIP")),
                None,
            )
            if match_link is None:
                avail = [
                    l.rsplit("%23", 2)[-2]
                    for l in links
                    if any(k in l.upper() for k in ("PREDISPATCH", "PDPASA"))
                    and l.upper().endswith(".ZIP")
                ]
                raise RuntimeError(
                    f"MMSDM archive: no file for '{search_key}' in {year}-{month:02d}. "
                    f"Available: {avail}"
                )

            file_url = f"https://nemweb.com.au{match_link}"
            print(f"    MMSDM archive: downloading {year}-{month:02d} ({search_key})...", flush=True)
            req = urllib.request.Request(file_url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=120) as resp:
                zdata = resp.read()

            # Build column want map: uppercase MMSDM col name → output col name
            want = {
                mmsdm_run_col_name.upper():      run_col,
                mmsdm_interval_col_name.upper(): interval_col,
                entity_col.upper():              entity_col,
            }
            for vc in value_cols:
                want[vc.upper()] = vc

            all_rows = []
            with zipfile.ZipFile(io.BytesIO(zdata)) as zf:
                for name in zf.namelist():
                    if not name.upper().endswith(".CSV"):
                        continue
                    with zf.open(name) as f:
                        text = f.read().decode("utf-8", errors="replace")
                    col_idx          = {}
                    intervention_idx = None
                    for row in _csv.reader(text.splitlines()):
                        if not row or len(row) < 5:
                            continue
                        rec = row[0].upper()
                        if rec == "I" and row[1].upper() == mmsdm_source_id:
                            hdrs             = [c.upper() for c in row[4:]]
                            col_idx          = {c: i for i, c in enumerate(hdrs)}
                            intervention_idx = col_idx.get("INTERVENTION")
                        elif rec == "D" and col_idx and row[1].upper() == mmsdm_source_id:
                            vals = row[4:]
                            # Skip intervention runs where the column exists
                            if intervention_idx is not None and intervention_idx < len(vals):
                                if vals[intervention_idx].strip() != "0":
                                    continue
                            r = {}
                            for up_col, out_col in want.items():
                                if up_col in col_idx and col_idx[up_col] < len(vals):
                                    r[out_col] = vals[col_idx[up_col]]
                            if len(r) == len(want):
                                all_rows.append(r)

            if not all_rows:
                raise RuntimeError(
                    f"MMSDM archive: no rows parsed for '{search_key}' in {year}-{month:02d}"
                )

            df = pd.DataFrame(all_rows)
            if floor_run:
                # Floor LASTCHANGED to nearest 30-min to approximate run boundaries
                df[run_col] = pd.to_datetime(df[run_col], format="mixed").dt.floor("30min")
            print(f"    MMSDM archive: parsed {len(df):,} rows.", flush=True)
            return df

        def _clean_up(API_response):

            # Handle conversion of data types
            API_response[run_col]      = pd.to_datetime(API_response[run_col], format="mixed")
            API_response[interval_col] = pd.to_datetime(API_response[interval_col], format="mixed")
            for col in value_cols:
                API_response[col] = pd.to_numeric(API_response[col], errors="coerce")

            # Build 5-min output grid indexed by delivery timestamp
            output_idx = pd.date_range(
                start[:16], end[:16],
                freq="5min", name="SETTLEMENTDATE"
            )
            # Extend T_df back to `start` (1 day before output window) so that runs
            # are included in the pivot — their h57+ values can then
            # be ffilled forward into the Jan 1 rows
            full_idx = pd.date_range(
                pd.Timestamp(modified_start[:16]), pd.Timestamp(end[:16]),
                freq="5min", name="SETTLEMENTDATE"
            )
            T_df = pd.DataFrame({"SETTLEMENTDATE": full_idx})

            # For each delivery timestamp T, find the most recently published run (run_time <= T)
            # This is the causally correct assignment: no future run information leaks into T
            run_times_df = (
                API_response[[run_col]]
                .drop_duplicates()
                .sort_values(run_col)
                .reset_index(drop=True)
            )
            T_df = pd.merge_asof(
                T_df, run_times_df,
                left_on="SETTLEMENTDATE", right_on=run_col,
                direction="backward"
            )

            # Join each T to all forecast rows from its assigned run
            merged = T_df.merge(API_response, on=run_col, how="left")

            # Drop T values where no prior run exists (no run available before T)
            merged = merged[merged[interval_col].notna()]

            # Compute horizon relative to delivery timestamp T using ceiling division:
            # h=1 → first 30-min period strictly after T, h=2 → second, etc.
            # ceil is required (not round) so that e.g. T=00:15 with interval=00:30
            # gives (900s / 1800) = 0.5 → ceil → 1, not round → 0 (which would drop it)
            merged["horizon"] = (
                (merged[interval_col] - merged["SETTLEMENTDATE"]).dt.total_seconds()
                .div(1800)
                .apply(math.ceil)
            )

            # Keep only forward-looking horizons
            merged = merged[merged["horizon"] >= 1]

            # Pivot each value column separately and concat
            frames = []
            for col in value_cols:
                pivot = (
                    merged.groupby(["SETTLEMENTDATE", entity_col, "horizon"])[col]
                    .mean()
                    .unstack([entity_col, "horizon"])
                )
                pivot = pivot.sort_index(axis=1)

                if entity_col == "REGIONID":
                    # Strip trailing digit: "NSW1" → "nsw"
                    pivot.columns = [
                        f"{nemseer_forecast_type.lower()}_{col.lower()}_{ent[:-1].lower()}_h{h}"
                        for ent, h in pivot.columns
                    ]
                else:
                    # Interconnector: replace dashes with underscores, e.g. "NSW1-QLD1" → "nsw1_qld1"
                    pivot.columns = [
                        f"{nemseer_forecast_type.lower()}_{col.lower()}_{ent.lower().replace('-', '_')}_h{h}"
                        for ent, h in pivot.columns
                    ]
                frames.append(pivot)

            result = pd.concat(frames, axis=1)
            result.index.name = "Date"

            # Reindex to the full 5-min grid, then ffill per horizon column:
            # carries the last known forecast for each horizon forward in time — no leakage
            # since ffill only looks backward (earlier timestamps)
            # full_idx covers the 3-day lookback so Dec 31 h57+ values ffill into Jan 1
            result = result.reindex(full_idx).ffill().reindex(output_idx)

            return result

        if _use_mmsdm:
            data = _API_call_mmsdm()
        elif _use_backup:
            data = _API_call_backup()
        else:
            data = _API_call_default()
        data_clean = _clean_up(data)
        return data_clean

    def _repeat_logic_backup(start: str, end: str):
        try:
            return _repeat_logic(start, end, _use_backup=True)
        except RuntimeError as e:
            print(f"  Per-run archive failed — trying MMSDM monthly archive...", flush=True)
            return _repeat_logic(start, end, _use_mmsdm=True)

    _month_compiler(
        datasource_fetch_function = _repeat_logic,
        start_date = start_date,
        end_date = end_date,
        datasource_file_path = datasource_file_path,
        fallback_fetch_function = _repeat_logic_backup
    )


In [11]:
"""
Datasource 8.1
"""
def _bid_availability(start: str, end: str) -> pd.DataFrame:
    BAND_COLS = [f"BANDAVAIL{i}" for i in range(1, 11)]

    def _API_call():
        return nemosis.dynamic_data_compiler(
            start_time=start, end_time=end,
            table_name="BIDPEROFFER_D",
            raw_data_location="Pre_processing/temporary_cache",
            select_columns=["INTERVAL_DATETIME", "DUID", "BIDTYPE", "MAXAVAIL"] + BAND_COLS,
            filter_cols=["BIDTYPE"], filter_values=[["ENERGY"]],
            fformat="feather", keep_csv=False, parse_data_types=False,
        )

    def _clean_up(API_response):
        API_response.columns = [c.upper() for c in API_response.columns]
        API_response = API_response.rename(columns={"INTERVAL_DATETIME": "SETTLEMENTDATE"})
        API_response["SETTLEMENTDATE"] = pd.to_datetime(API_response["SETTLEMENTDATE"])

        num_cols = ["MAXAVAIL"] + BAND_COLS
        API_response[num_cols] = API_response[num_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)

        agg = API_response.groupby(["SETTLEMENTDATE", "DUID"])[num_cols].mean()

        # {DUID}_maxavail columns
        maxavail = agg["MAXAVAIL"].unstack("DUID")
        maxavail.columns = [f"{duid}_maxavail" for duid in maxavail.columns]
        maxavail = maxavail.resample("5min").mean().fillna(0)

        # {DUID}_bands — resample numerically first, then convert to comma-separated strings
        # This avoids slow string-based resampling
        band_data = agg[BAND_COLS].unstack("DUID").resample("5min").last()

        duids = sorted(band_data.columns.get_level_values("DUID").unique())
        bands_dict = {}
        for duid in duids:
            duid_cols = band_data.xs(duid, axis=1, level="DUID").fillna(0).astype(int).astype(str)
            bands_dict[f"{duid}_bands"] = duid_cols.iloc[:, 0].str.cat(duid_cols.iloc[:, 1:], sep=",")
        bands = pd.DataFrame(bands_dict, index=band_data.index)

        result = pd.concat([maxavail, bands], axis=1)

        all_duids = sorted(c.rsplit("_maxavail", 1)[0] for c in maxavail.columns)
        ordered = [col for duid in all_duids for col in (f"{duid}_maxavail", f"{duid}_bands") if col in result.columns]
        result = result[ordered]
        result.index.name = "Date"
        return result

    data = _API_call()
    return _clean_up(data)

In [12]:
"""
Datasource 8.2
"""
def _bid_prices(start: str, end: str) -> pd.DataFrame:
    PRICE_COLS = [f"PRICEBAND{i}" for i in range(1, 11)]

    def _API_call():
        return nemosis.dynamic_data_compiler(
            start_time=start, end_time=end,
            table_name="BIDDAYOFFER_D",
            raw_data_location="Pre_processing/temporary_cache",
            select_columns=["SETTLEMENTDATE", "DUID", "BIDTYPE"] + PRICE_COLS,
            filter_cols=["BIDTYPE"], filter_values=[["ENERGY"]],
            fformat="feather", keep_csv=False, parse_data_types=False,
        )

    def _clean_up(API_response):
        API_response.columns = [c.upper() for c in API_response.columns]
        API_response["SETTLEMENTDATE"] = pd.to_datetime(API_response["SETTLEMENTDATE"])
        API_response[PRICE_COLS] = API_response[PRICE_COLS].apply(pd.to_numeric, errors="coerce").fillna(0.0)

        agg = API_response.groupby(["SETTLEMENTDATE", "DUID"])[PRICE_COLS].mean()

        # Resample numerically first (ffill daily→5min), then convert to comma-separated strings
        price_data = agg.unstack("DUID").resample("5min").ffill()

        duids = sorted(price_data.columns.get_level_values("DUID").unique())
        prices_dict = {}
        for duid in duids:
            duid_cols = price_data.xs(duid, axis=1, level="DUID").fillna(0).astype(int).astype(str)
            prices_dict[f"{duid}_prices"] = duid_cols.iloc[:, 0].str.cat(duid_cols.iloc[:, 1:], sep=",")
        result = pd.DataFrame(prices_dict, index=price_data.index)

        result.index.name = "Date"
        return result

    data = _API_call()
    return _clean_up(data)


In [13]:
"""
Datasource 1

 - Datasource origin: nemosis
 - Datasource name: dispatch price     : AEMO's DISPATCHPRICE table — contains the Regional Reference Price (RRP) and related pricing outcomes published for each region at the end of every 5-minute dispatch interval
 - Variables:
    RRP         : Regional Reference Price ($/MWh) — the spot price for electricity in each NEM region, set by the market clearing process for each 5-minute dispatch interval
    REGIONID    : NEM region identifier used to filter and pivot the data (e.g. NSW1, QLD1, VIC1, SA1)
"""
if False:
    _month_compiler(
        datasource_fetch_function = _dispatch_price,
        start_date = pd.Timestamp("2018/01/01"),
        end_date = pd.Timestamp("2026/02/01"),
        datasource_file_path = Path("Processed_data/1_dispatch_price.parquet")
    )

  Found 97 already-processed month(s) — will skip.
  1/97 2018-01 skipping (already processed).
  2/97 2018-02 skipping (already processed).
  3/97 2018-03 skipping (already processed).
  4/97 2018-04 skipping (already processed).
  5/97 2018-05 skipping (already processed).
  6/97 2018-06 skipping (already processed).
  7/97 2018-07 skipping (already processed).
  8/97 2018-08 skipping (already processed).
  9/97 2018-09 skipping (already processed).
 10/97 2018-10 skipping (already processed).
 11/97 2018-11 skipping (already processed).
 12/97 2018-12 skipping (already processed).
 13/97 2019-01 skipping (already processed).
 14/97 2019-02 skipping (already processed).
 15/97 2019-03 skipping (already processed).
 16/97 2019-04 skipping (already processed).
 17/97 2019-05 skipping (already processed).
 18/97 2019-06 skipping (already processed).
 19/97 2019-07 skipping (already processed).
 20/97 2019-08 skipping (already processed).
 21/97 2019-09 skipping (already processed).
 22/

In [14]:
"""
Datasource 2

 - Datasource origin: nemosis
 - Datasource name: dispatch region sum    : AEMO's DISPATCHREGIONSUM table — contains region-level demand, generation, and interconnector summary statistics published at the end of every 5-minute dispatch interval
 - Variables:
    TOTALDEMAND               : Total metered demand (MW) for the region in the dispatch interval
    AVAILABLEGENERATION       : Total available generation capacity (MW) that could be dispatched in the region
    NETINTERCHANGE            : Net interconnector flow (MW) into the region — positive means imports, negative means exports
    DEMANDFORECAST            : AEMO's forecast of regional demand (MW) for the dispatch interval
    DISPATCHABLEGENERATION    : Total generation (MW) actually dispatched in the region for the interval

    REGIONID                  : NEM region identifier used to filter and pivot the data (e.g. NSW1, QLD1, VIC1, SA1)
"""
if False:
    _month_compiler(
        datasource_fetch_function = _dispatch_region_sum,
        start_date = pd.Timestamp("2018/01/01"),
        end_date = pd.Timestamp("2026/02/01"),
        datasource_file_path = Path("Processed_data/2_dispatch_region_sum.parquet")
    )

In [15]:
"""
Datasource 3
 - Datasource origin: nemosis
 - Datasource name: generation fuel   : Per-fuel-type aggregated generation (MW) from AEMO's DISPATCH_UNIT_SCADA table,
                                         with DUID-to-fuel mapping sourced from the NEM Registration and Exemption List.
 - Variables:
    coal_mw_{region}              : Total coal generation (MW)
    wind_mw_{region}              : Total wind generation (MW)
    solar_mw_{region}             : Total solar/PV generation (MW)
    hydro_mw_{region}             : Total hydro generation (MW)
    gas_mw_{region}               : Total gas/diesel/distillate generation (MW)
    battery_charge_mw_{region}    : Total battery charging (MW, positive)
    battery_discharge_mw_{region} : Total battery discharging (MW, positive)
"""
if False:
    _month_compiler(
        datasource_fetch_function = _generation_fuel,
        start_date = pd.Timestamp("2018/01/01"),
        end_date = pd.Timestamp("2026/02/01"),
        datasource_file_path = Path("Processed_data/3_generation_fuel.parquet")
    )


In [16]:
"""
Datasource 4

 - Datasource origin: www.aemo.com.au
 - Datasource name: STTM_DWGM
    STTM   : AEMO's Short Term Trading Market — daily ex ante commodity price ($/GJ) at each gas hub, set by the market clearing process for the upcoming gas day
    DWGM   : AEMO's Declared Wholesale Gas Market — 5-interval-per-day scheduled price ($/GJ) for Victoria; each interval covers a ~4-hour scheduling horizon (6am, 10am, 2pm, 6pm, 10pm)
 - Variables:
    gas_price_nsw : STTM Sydney hub ex ante daily commodity price ($/GJ)
    gas_price_qld : STTM Brisbane hub ex ante daily commodity price ($/GJ)
    gas_price_sa  : STTM Adelaide hub ex ante daily commodity price ($/GJ)
    gas_price_vic : DWGM Victoria scheduled price ($/GJ) — 5 intervals per gas day, forward-filled within each ~4-hour scheduling horizon
"""
if False:
    _one_shot_compiler(
        datasource_fetch_function = _STTM_DWGM,
        start_date = pd.Timestamp("2018/01/01"),
        end_date = pd.Timestamp("2026/02/01"),
        datasource_file_path = Path("Processed_data/4_STTM_DWGM.parquet")
    )

In [17]:
"""
Datasource 5

 - Datasource origin: www.visualcrossing.com
 - Datasource name: weather
 - Variables:
    temp             : Mean air temperature (°C)
    feelslike        : Apparent temperature combining heat index and wind chill (°C)
    dew              : Dew point temperature — measure of atmospheric moisture (°C)
    humidity         : Relative humidity (%)
    precip           : Total precipitation (rain/snow liquid equivalent) for the period (mm)
    precipprob       : Probability of precipitation (%)
    preciptype       : Type(s) of precipitation (rain, snow, freezing rain, ice)
    snow             : New snowfall amount (cm)
    snowdepth        : Depth of snow currently on the ground (cm)
    windgust         : Maximum short-term wind gust speed (kph)
    windspeed        : Mean wind speed (kph)
    winddir          : Wind direction — degrees clockwise from north (°)
    sealevelpressure : Atmospheric pressure normalised to sea level (mb)
    cloudcover       : Fraction of sky covered by cloud (%)
    visibility       : Horizontal visibility distance (km)
    solarradiation   : Instantaneous solar radiation at the surface (W/m²)
    solarenergy      : Total solar energy accumulated over the period (MJ/m²)
    uvindex          : UV exposure index (0–10)
    severerisk       : Probability of severe weather events (%)
    conditions       : Short text summary of weather conditions
    icon             : Weather icon identifier
    stations         : Weather station(s) used as source for the observation
"""
if False:
    _one_shot_compiler(
        datasource_fetch_function = _weather,
        start_date = pd.Timestamp("2018/01/01"),
        end_date = pd.Timestamp("2026/02/01"),
        datasource_file_path = Path("Processed_data/5_weather.parquet")
    )

 2022-09 saved & raw files deleted.
 58/96 2022-10 fetching...

In [18]:
"""
Datasource 6.1
 - Datasource origin: nemseer
 - Datasource name: predispatch price  : AEMO's PREDISPATCH PRICE table — contains 30-minute pre-dispatch pricing forecasts for each NEM region, published every half hour
 - Variables:
    RRP : Regional Reference Price forecast ($/MWh)
"""

if False:
    _nemseer_pull(
        start_date = pd.Timestamp("2018/01/01"),
        end_date = pd.Timestamp("2026/02/01"),
        datasource_file_path = Path("Processed_data/6_1_predispatch_price.parquet"),
        nemseer_forecast_type = "PREDISPATCH",
        nemseer_table_name = "PRICE",
        value_cols = ["RRP"],
        run_col = "PREDISPATCH_RUN_DATETIME",
        interval_col = "DATETIME",
        entity_col = "REGIONID",
    )


In [19]:
"""
Datasource 6.2
 - Datasource origin: nemseer
 - Datasource name: predispatch region sum  : AEMO's PREDISPATCH REGIONSUM table — contains 30-minute pre-dispatch demand and generation forecasts for each NEM region, published every half hour
 - Variables:
    TOTALDEMAND         : Forecast total regional demand (MW)
    AVAILABLEGENERATION : Forecast available generation capacity (MW)
    NETINTERCHANGE      : Forecast net interconnector flow (MW)
    DEMANDFORECAST      : AEMO's internal demand forecast (MW)
    AVAILABLELOAD       : Forecast available scheduled load (MW)
"""

if True:
    if False:
        _nemseer_pull(
            start_date = pd.Timestamp("2018/01/01"),
            end_date = pd.Timestamp("2026/02/01"),
            datasource_file_path = Path("Processed_data/6_2_1_predispatch_regionsum.parquet"),
            nemseer_forecast_type = "PREDISPATCH",
            nemseer_table_name = "REGIONSUM",
            value_cols = ["TOTALDEMAND"],
            run_col = "PREDISPATCH_RUN_DATETIME",
            interval_col = "DATETIME",
            entity_col = "REGIONID",
        )
    if False:
        _nemseer_pull(
            start_date = pd.Timestamp("2018/01/01"),
            end_date = pd.Timestamp("2026/02/01"),
            datasource_file_path = Path("Processed_data/6_2_2_predispatch_regionsum.parquet"),
            nemseer_forecast_type = "PREDISPATCH",
            nemseer_table_name = "REGIONSUM",
            value_cols = ["AVAILABLEGENERATION"],
            run_col = "PREDISPATCH_RUN_DATETIME",
            interval_col = "DATETIME",
            entity_col = "REGIONID",
        )
    if False:
        _nemseer_pull(
            start_date = pd.Timestamp("2018/01/01"),
            end_date = pd.Timestamp("2026/02/01"),
            datasource_file_path = Path("Processed_data/6_2_3_predispatch_regionsum.parquet"),
            nemseer_forecast_type = "PREDISPATCH",
            nemseer_table_name = "REGIONSUM",
            value_cols = ["NETINTERCHANGE"],
            run_col = "PREDISPATCH_RUN_DATETIME",
            interval_col = "DATETIME",
            entity_col = "REGIONID",
        )
    if False:
        _nemseer_pull(
            start_date = pd.Timestamp("2018/01/01"),
            end_date = pd.Timestamp("2026/02/01"),
            datasource_file_path = Path("Processed_data/6_2_4_predispatch_regionsum.parquet"),
            nemseer_forecast_type = "PREDISPATCH",
            nemseer_table_name = "REGIONSUM",
            value_cols = ["DEMANDFORECAST"],
            run_col = "PREDISPATCH_RUN_DATETIME",
            interval_col = "DATETIME",
            entity_col = "REGIONID",
        )
    if False:
        _nemseer_pull(
            start_date = pd.Timestamp("2018/01/01"),
            end_date = pd.Timestamp("2026/02/01"),
            datasource_file_path = Path("Processed_data/6_2_5_predispatch_regionsum.parquet"),
            nemseer_forecast_type = "PREDISPATCH",
            nemseer_table_name = "REGIONSUM",
            value_cols = ["AVAILABLELOAD"],
            run_col = "PREDISPATCH_RUN_DATETIME",
            interval_col = "DATETIME",
            entity_col = "REGIONID",
        )

In [20]:

"""
Datasource 6.3
 - Datasource origin: nemseer
 - Datasource name: predispatch interconnector solution  : AEMO's PREDISPATCH INTERCONNECTORRES table — contains 30-minute pre-dispatch MW flow forecasts for each NEM interconnector, published every half hour
 - Variables:
    MWFLOW        : Forecast MW flow on each interconnector
    METEREDMWFLOW : Metered MW flow forecast
"""

if True:
    if False:
        _nemseer_pull(
            start_date = pd.Timestamp("2018/01/01"),
            end_date = pd.Timestamp("2026/02/01"),
            datasource_file_path = Path("Processed_data/6_3_1_predispatch_interconnectorsoln.parquet"),
            nemseer_forecast_type = "PREDISPATCH",
            nemseer_table_name = "INTERCONNECTORRES",
            value_cols = ["MWFLOW"],
            run_col = "PREDISPATCH_RUN_DATETIME",
            interval_col = "DATETIME",
            entity_col = "INTERCONNECTORID",
        )
    if False:
        _nemseer_pull(
            start_date = pd.Timestamp("2018/01/01"),
            end_date = pd.Timestamp("2026/02/01"),
            datasource_file_path = Path("Processed_data/6_3_2_predispatch_interconnectorsoln.parquet"),
            nemseer_forecast_type = "PREDISPATCH",
            nemseer_table_name = "INTERCONNECTORRES",
            value_cols = ["METEREDMWFLOW"],
            run_col = "PREDISPATCH_RUN_DATETIME",
            interval_col = "DATETIME",
            entity_col = "INTERCONNECTORID",
        )


In [21]:
"""
Datasource 7
 - Datasource origin: nemseer
 - Datasource name: pdpasa region solution  : AEMO's PDPASA REGIONSOLUTION table — contains 30-minute demand and reserve forecasts for each NEM region, published every half hour
 - Variables:
    DEMAND10                  : 10th percentile demand forecast (MW)
    DEMAND50                  : 50th percentile demand forecast (MW)
    DEMAND90                  : 90th percentile demand forecast (MW)
    RESERVEREQ                : Reserve requirement (MW)
    SURPLUSRESERVE            : Surplus reserve above requirement (MW)
    MAXSURPLUSRESERVE         : Maximum surplus reserve (MW)
    MAXSPARECAPACITY          : Maximum spare capacity headroom (MW)
    AGGREGATEPASAAVAILABILITY : Sum of PASA availability across all scheduled generators + semi-scheduled UIGF (MW)
"""

if True:
    if False:
        _nemseer_pull(
            start_date = pd.Timestamp("2018/01/01"),
            end_date = pd.Timestamp("2026/02/01"),
            datasource_file_path = Path("Processed_data/7_1_1_pdpasa_regionsolution.parquet"),
            nemseer_forecast_type = "PDPASA",
            nemseer_table_name = "REGIONSOLUTION",
            value_cols = ["DEMAND10"],
            run_col = "RUN_DATETIME",
            interval_col = "INTERVAL_DATETIME",
            entity_col = "REGIONID",
        )
    if False:
        _nemseer_pull(
            start_date = pd.Timestamp("2018/01/01"),
            end_date = pd.Timestamp("2026/02/01"),
            datasource_file_path = Path("Processed_data/7_1_2_pdpasa_regionsolution.parquet"),
            nemseer_forecast_type = "PDPASA",
            nemseer_table_name = "REGIONSOLUTION",
            value_cols = ["DEMAND50"],
            run_col = "RUN_DATETIME",
            interval_col = "INTERVAL_DATETIME",
            entity_col = "REGIONID",
        )
    if False:
        _nemseer_pull(
            start_date = pd.Timestamp("2018/01/01"),
            end_date = pd.Timestamp("2026/02/01"),
            datasource_file_path = Path("Processed_data/7_1_3_pdpasa_regionsolution.parquet"),
            nemseer_forecast_type = "PDPASA",
            nemseer_table_name = "REGIONSOLUTION",
            value_cols = ["DEMAND90"],
            run_col = "RUN_DATETIME",
            interval_col = "INTERVAL_DATETIME",
            entity_col = "REGIONID",
        )
    if False:
        _nemseer_pull(
            start_date = pd.Timestamp("2018/01/01"),
            end_date = pd.Timestamp("2026/02/01"),
            datasource_file_path = Path("Processed_data/7_1_4_pdpasa_regionsolution.parquet"),
            nemseer_forecast_type = "PDPASA",
            nemseer_table_name = "REGIONSOLUTION",
            value_cols = ["RESERVEREQ"],
            run_col = "RUN_DATETIME",
            interval_col = "INTERVAL_DATETIME",
            entity_col = "REGIONID",
        )
    if False:
        _nemseer_pull(
            start_date = pd.Timestamp("2018/01/01"),
            end_date = pd.Timestamp("2026/02/01"),
            datasource_file_path = Path("Processed_data/7_1_5_pdpasa_regionsolution.parquet"),
            nemseer_forecast_type = "PDPASA",
            nemseer_table_name = "REGIONSOLUTION",
            value_cols = ["SURPLUSRESERVE"],
            run_col = "RUN_DATETIME",
            interval_col = "INTERVAL_DATETIME",
            entity_col = "REGIONID",
        )
    if False:
        _nemseer_pull(
            start_date = pd.Timestamp("2018/01/01"),
            end_date = pd.Timestamp("2026/02/01"),
            datasource_file_path = Path("Processed_data/7_1_6_pdpasa_regionsolution.parquet"),
            nemseer_forecast_type = "PDPASA",
            nemseer_table_name = "REGIONSOLUTION",
            value_cols = ["MAXSURPLUSRESERVE"],
            run_col = "RUN_DATETIME",
            interval_col = "INTERVAL_DATETIME",
            entity_col = "REGIONID",
        )
    if False:
        _nemseer_pull(
            start_date = pd.Timestamp("2018/01/01"),
            end_date = pd.Timestamp("2026/02/01"),
            datasource_file_path = Path("Processed_data/7_1_7_pdpasa_regionsolution.parquet"),
            nemseer_forecast_type = "PDPASA",
            nemseer_table_name = "REGIONSOLUTION",
            value_cols = ["MAXSPARECAPACITY"],
            run_col = "RUN_DATETIME",
            interval_col = "INTERVAL_DATETIME",
            entity_col = "REGIONID",
        )
    if False:
        _nemseer_pull(
            start_date = pd.Timestamp("2018/01/01"),
            end_date = pd.Timestamp("2026/02/01"),
            datasource_file_path = Path("Processed_data/7_1_8_pdpasa_regionsolution.parquet"),
            nemseer_forecast_type = "PDPASA",
            nemseer_table_name = "REGIONSOLUTION",
            value_cols = ["AGGREGATEPASAAVAILABILITY"],
            run_col = "RUN_DATETIME",
            interval_col = "INTERVAL_DATETIME",
            entity_col = "REGIONID",
        )


In [22]:
"""
Datasource 8.1

 - Datasource origin: nemosis 
 - Datasource name: bid stack   : Per-DUID bid availability from AEMO's BIDPEROFFER_D table.
 - Variables (per DUID):
    {DUID}_maxavail : Total maximum available capacity (MW) for the dispatch interval
    {DUID}_bands    : Comma-separated string of the 10 band MW offers e.g. "22,0,150,0,..."
"""
if False:
    _month_compiler(
        datasource_fetch_function = _bid_availability,
        start_date = pd.Timestamp("2018/01/01"),
        end_date = pd.Timestamp("2026/02/01"),
        datasource_file_path = Path("Processed_data/8_bid_availability.parquet")
    )

"""
Datasource 8.2

 - Datasource origin: nemosis 
 - Datasource name: bid prices  : Per-DUID daily price band offers from AEMO's BIDDAYOFFER_D table.
 - Variables (per DUID):
    {DUID}_prices : Comma-separated string of the 10 price band offers ($/MWh) e.g. "0,100,300,..."
"""
if False:
    _month_compiler(
        datasource_fetch_function = _bid_prices,
        start_date = pd.Timestamp("2018/01/01"),
        end_date = pd.Timestamp("2026/02/01"),
        datasource_file_path = Path("Processed_data/8_bid_prices.parquet")
    )


In [23]:
%reset -f